In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

def smiles_to_3d_rdkit_fast(smiles, out_xyz='molecule_opt.xyz', max_attempts=3):
    """
    Быстрая 3D оптимизация с обработкой металлоорганики.
    
    Parameters:
    -----------
    smiles : str
        SMILES строка молекулы
    out_xyz : str
        Имя выходного файла для оптимизированной структуры
    max_attempts : int
        Максимальное количество попыток генерации конформера
    
    Returns:
    --------
    bool : True если успешно, False если ошибка
    """
    try:
        # Создаем RDKit молекулу из SMILES
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return False
        
        mol = Chem.AddHs(mol)
        
        # Генерируем 3D конформер с минимальными попытками
        success = False
        for attempt in range(max_attempts):
            try:
                # Используем более быстрые параметры
                params = AllChem.ETKDG()
                params.randomSeed = 42 + attempt
                params.useRandomCoords = True  # Быстрее для сложных молекул
                
                result = AllChem.EmbedMolecule(mol, params)
                
                if result == 0:
                    success = True
                    break
            except:
                # Если ETKDG не сработал, пробуем простой метод
                try:
                    result = AllChem.EmbedMolecule(mol, randomSeed=42 + attempt, 
                                                maxAttempts=100, 
                                                useRandomCoords=True)
                    if result == 0:
                        success = True
                        break
                except:
                    continue
        
        if not success:
            return False
        
        # Оптимизируем молекулу - используем MMFF если UFF не подходит для металлов
        try:
            # Проверяем, содержит ли молекула металлы
            has_metals = any(atom.GetSymbol() in ['Ir', 'Pt', 'Pd', 'Rh', 'Ru', 'Os', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ag', 'Au', 'Cd', 'Hg', 'Mn', 'Mo', 'W', 'Cr', 'V', 'Ti', 'Sc', 'Y', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu'] for atom in mol.GetAtoms())
            
            if has_metals:
                # Для металлоорганики используем MMFF (он лучше поддерживает металлы)
                try:
                    AllChem.MMFFOptimizeMolecule(mol, maxIters=2)  # Меньше итераций для скорости
                except:
                    # Если MMFF не сработал, сохраняем без оптимизации
                    pass
            else:
                # Для органики используем UFF
                try:
                    AllChem.UFFOptimizeMolecule(mol, maxIters=2)  # Меньше итераций для скорости
                except:
                    # Если UFF не сработал, сохраняем без оптимизации
                    pass
        except:
            # Если оптимизация не удалась, просто сохраняем текущую структуру
            pass
        
        # Сохраняем XYZ файл
        try:
            with open(out_xyz, 'w') as f:
                f.write(Chem.MolToXYZBlock(mol))
            return True
        except:
            return False
        
    except Exception as e:
        return False

def optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes_rdkit', start_index=0, batch_size=100):
    """
    Оптимизирует все молекулы из датафрейма с улучшенной производительностью.
    
    Parameters:
    -----------
    df : DataFrame
        Датафрейм с колонкой SMILES
    smiles_column : str
        Название колонки со SMILES
    output_dir : str
        Директория для сохранения оптимизированных структур
    start_index : int
        Порядковый номер строки, с которой начинается расчет
    batch_size : int
        Размер батча для прогресс-бара (для более частого обновления)
    
    Returns:
    --------
    DataFrame : Обновленный датафрейм с колонкой путей к файлам
    """
    # Создаем директорию для сохранения
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*100)
    print(f"БЫСТРАЯ ОПТИМИЗАЦИЯ КОМПЛЕКСОВ ИЗ ДАТАФРЕЙМА (RDKit)")
    print("="*100)
    print(f"Всего строк в датафрейме: {len(df)}")
    print(f"Начальный индекс: {start_index}")
    print(f"Ожидаемое количество молекул: {len(df) - start_index}")
    print()
    
    # Добавляем колонку для путей к файлам
    if 'Complex_XYZ_File' not in df.columns:
        df['Complex_XYZ_File'] = None
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    empty_count = 0
    
    # Подсчитываем общее количество для обработки
    total_to_process = 0
    for idx in range(start_index, len(df)):
        smiles = df.iloc[idx][smiles_column]
        if not pd.isna(smiles) and smiles != '' and not os.path.exists(os.path.join(output_dir, f"complex_{idx}.xyz")):
            total_to_process += 1
    
    print(f"Нужно обработать: {total_to_process} молекул")
    print()
    
    # Используем tqdm для прогресс-бара с меньшими батчами для частого обновления
    processed_idx = start_index
    while processed_idx < len(df):
        # Проверяем SMILES
        smiles = df.iloc[processed_idx][smiles_column]
        
        if pd.isna(smiles) or smiles == '':
            empty_count += 1
            processed_idx += 1
            continue
        
        # Имя файла: порядковый номер
        out_file = os.path.join(output_dir, f"complex_{processed_idx}.xyz")
        
        # Проверяем, не оптимизирован ли уже
        if os.path.exists(out_file):
            df.at[processed_idx, 'Complex_XYZ_File'] = out_file
            skip_count += 1
            processed_idx += 1
            continue
        
        # Оптимизируем структуру
        success = smiles_to_3d_rdkit_fast(smiles, out_file)
        
        if success:
            df.at[processed_idx, 'Complex_XYZ_File'] = out_file
            success_count += 1
        else:
            fail_count += 1
        
        processed_idx += 1
        
        # Обновляем прогресс бар каждые 10 молекул
        if processed_idx % 10 == 0 or processed_idx >= len(df):
            current_progress = processed_idx - start_index - skip_count - empty_count - fail_count
            tqdm.write(f"\rОбработано: {current_progress}/{total_to_process} ({current_progress/max(total_to_process,1)*100:.1f}%)", end='')
    
    print()  # Новая строка после прогресса
    print()
    print("="*100)
    print("СТАТИСТИКА ОПТИМИЗАЦИИ:")
    print(f"  ✓ Успешно оптимизировано: {success_count}")
    print(f"  ⏭️  Пропущено (уже есть): {skip_count}")
    print(f"  ❌ Ошибок оптимизации: {fail_count}")
    print(f"  ⚠️  Пустых SMILES: {empty_count}")
    print(f"  📁 Файлы сохранены в: {output_dir}/")
    print("="*100)
    
    return df

# Укажите начальный индекс здесь
i = 5000 # измените это значение на нужный начальный индекс

import pandas as pd

df = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/dataset.csv', sep=';')

# Оптимизируем все комплексы из датафрейма, начиная с индекса i
df = optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes_rdkit', start_index=i)

# # Сохраняем обновленный датафрейм
# df.to_csv('processed_complexes_with_xyz.csv', index=False, sep=';')

# Выводим примеры
print("\nПримеры обработанных строк:")
print(df[['SMILES', 'Complex_XYZ_File']].iloc[i:i+10])

# Выводим статистику по типам молекул
print(f"\nОбщая статистика:")
print(f"Всего обработано: {len(df[df['Complex_XYZ_File'].notna()])}")
print(f"Процент успешных: {len(df[df['Complex_XYZ_File'].notna()])/len(df)*100:.2f}%")

[18:43:54] Interrupted, cancelling conformer generation
[18:43:54] UFFTYPER: Unrecognized atom type: Pt3+2 (50)
